# RT · **BENCHMARK** · all generators

Scores every pretrained checkpoint on **held-out RelBench v2 tasks**, with no fine-tuning, across a
sweep of context lengths. This is study **RQ1** ("which generator gives the most accurate learning
to an RFM model?").

## Which databases, and why

RelBench **v2** has 11 `rel-*` databases. GRDM and RelDiff build their corpora from the 35% temporal
slice of **rel-f1, rel-hm, rel-avito, rel-event, rel-trial**, so scoring those arms there would be
scoring them on data derived from their own training material. Removing those five leaves **six
untouched databases**:

`rel-amazon` · `rel-arxiv` · `rel-mimic` · `rel-ratebeer` · `rel-salt` · `rel-stack`

Of their tasks RT can score only **binary classification** (→ AUROC) and **regression** (→ R²) --
its decoders are boolean and numeric. Multiclass and link-prediction tasks have no matching head and
are listed in `bench.UNSCORABLE`. That leaves **16 tasks**, and note that **rel-salt contributes
none**: all eight of its tasks are multiclass.

## Context sweep

`100 · 200 · 512 · 1024`. RT uses no positional encodings, so any length is architecturally valid and
1024 is merely what pretraining used -- the sweep asks how much each arm exploits extra in-context
evidence.

**The 30000 sweep lives in a separate notebook**, `RT_Benchmark_30k.ipynb`. At that length the batch
size drops to 1 and the run is an order of magnitude slower with partial OOMs expected, so keeping it
apart lets this sweep finish cleanly.

Each cell is capped by **predictions collected** (`MAX_SAMPLES`), not by batches. Batch size shrinks
as context grows, so a batch cap would quietly give the long-context runs fewer predictions and wider
error bars. Every row records `n`.

## Versions

This notebook installs **relbench 2.1.2** (the v2 datasets). PluRel pins relbench 1.1.0, but nothing
here needs PluRel installed -- `rt.model` and `rt.data` do not import relbench at all.


In [ ]:
#@title Config  (BENCHMARK · short + standard contexts)
# checkpoints to score: generator -> path on Drive
CHECKPOINTS = {
    "rdbpfn":  "/content/drive/MyDrive/rt_rdg/checkpoints/rdbpfn/rdbpfn_final.pt",
    "grdm":    "/content/drive/MyDrive/rt_rdg/checkpoints/grdm/grdm_final.pt",
    "reldiff": "/content/drive/MyDrive/rt_rdg/checkpoints/reldiff/reldiff_final.pt",
    "plurel":  "/content/drive/MyDrive/rt_rdg/checkpoints/plurel/plurel_final.pt",
}

CTX_LENS    = (100, 200, 512, 1024)
MAX_SAMPLES = 2048      # PREDICTIONS per (task, ctx), not batches -- keeps lengths comparable
NUM_WORKERS = 2
SPLIT       = "test"
RESULT_NAME = "rt_benchmark"

# --- memory ------------------------------------------------------------------------------
# Upstream builds each attention mask as a dense (B, S, S) tensor and then hands FlexAttention a
# mask_mod that just indexes it -- ~9.9 GB at ctx 30000 / batch 1. LEAN_MASKS hands FlexAttention
# the predicates directly, so no dense mask is ever allocated (~0.7 MB of BlockMask instead) and
# the sweep fits on a much smaller card. The preflight proves the two are elementwise identical
# before anything is scored; set this False to fall back to the stock path.
LEAN_MASKS  = True

# --- sharding ---------------------------------------------------------------------------
# The whole sweep in one session needs a long GPU booking. Slicing it by generator and/or
# database turns it into short independent jobs that can run on separate days or in parallel
# sessions -- each writes its own CSV and cell 6 merges them. Leave all three alone to run
# everything at once.
ONLY_GENERATORS = None   # e.g. ["rdbpfn"]     -- subset of CHECKPOINTS to score here

# The three databases whose prepared archives exist on Drive. rel-amazon and rel-mimic are left
# out because their preparation did not complete; adding either one back is just this list plus a
# rerun, and every result row records its database so the reduced set stays visible.
#   full held-out set: rel-amazon(5) rel-ratebeer(5) rel-stack(3) rel-arxiv(2) rel-mimic(1)
#   here:              rel-ratebeer(5) rel-stack(3) rel-arxiv(2) = 10 of 16 tasks
# Contamination is unaffected: no generator was built from any of these, so RQ1 still holds.
EVAL_DBS        = ["rel-ratebeer", "rel-stack", "rel-arxiv"]
SHARD           = "3db"  # names this shard's CSV; REQUIRED if either filter is set

DRIVE_DIR = "rt_rdg"
REPO      = "/content/plurel"
SEED      = 0

import os, time
from pathlib import Path

if ONLY_GENERATORS:
    missing = [g for g in ONLY_GENERATORS if g not in CHECKPOINTS]
    assert not missing, f"unknown generator(s) {missing}; known: {sorted(CHECKPOINTS)}"
    CHECKPOINTS = {g: CHECKPOINTS[g] for g in ONLY_GENERATORS}

# Without a distinct name every shard writes the same CSV on Drive and the last one wins,
# silently discarding the others. Fail here rather than after hours of GPU time.
assert SHARD or (not ONLY_GENERATORS and not EVAL_DBS), (
    "this session runs only part of the sweep, so set SHARD to a distinct name "
    "(e.g. SHARD = 'rdbpfn') -- otherwise it would overwrite the other shards' results")
RESULT_FILE = f"{RESULT_NAME}_{SHARD}" if SHARD else RESULT_NAME

print(f"{len(CHECKPOINTS)} checkpoints x {len(CTX_LENS)} context lengths"
      f"{' x ' + str(len(EVAL_DBS)) + ' databases' if EVAL_DBS else ''}")
print(f"results -> {RESULT_FILE}.csv")


## 1. Environment

In [ ]:
#@title Clone PluRel (for rt/ + rustler) + install deps
%cd /content
![ -d plurel ] || git clone --depth 1 https://github.com/snap-stanford/plurel.git

import sys, os
assert sys.version_info[:2] >= (3, 12), "rustler is built abi3-py312"
import numpy
open("/content/constraints.txt", "w").write(f"numpy=={numpy.__version__}\n")
C = "--constraint /content/constraints.txt"

# relbench 2.x for the v2 datasets. rt/ itself never imports relbench, so PluRel's 1.1.0 pin
# is irrelevant here -- we do not install the plurel package.
!pip -q install {C} "relbench==2.1.2" pytorch-frame sentence-transformers ml_dtypes \
    maturin maturin-import-hook strictfire orjson pyarrow duckdb pooch tqdm einops

sys.path.insert(0, "/content/plurel")
os.environ.setdefault("HOME", "/root")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from google.colab import drive
drive.mount("/content/drive")
PERSIST = Path(f"/content/drive/MyDrive/{DRIVE_DIR}")
(PERSIST / "benchmark").mkdir(parents=True, exist_ok=True)
print("results ->", PERSIST / "benchmark")


## 2. The `rt_icl` library

In [ ]:
#@title Write rt_icl (gzip+base64 embedded)
import base64, gzip, json, pathlib

PAYLOAD = "..."

pkg = pathlib.Path("/content/rt_icl")
pkg.mkdir(parents=True, exist_ok=True)
for name, text in json.loads(gzip.decompress(base64.b64decode(PAYLOAD)).decode("utf-8")).items():
    (pkg / name).write_text(text, encoding="utf-8")
if "/content" not in sys.path:
    sys.path.insert(0, "/content")

import rt_icl
from rt_icl import bench, core, adapters, compat, pipeline, prep, train
print("rt_icl", rt_icl.__version__, "->", sorted(p.name for p in pkg.glob("*.py")))


## 3. Preflight

In [ ]:
#@title Preflight: GPU · file descriptors · rustler · checkpoints
gpu = prep.check_gpu()
print("GPU:", gpu)
assert gpu["cuda"], "No CUDA device. Runtime -> Change runtime type -> GPU."
# `bench.load_model` casts the checkpoint to bfloat16 on CUDA, so this applies to EVERY sweep,
# not only the 30k one. LEAN_MASKS cuts the MEMORY needed, not the requirement for Ampere: bf16
# and FlexAttention's kernels both need compute >= 8.0, and scoring one arm in another dtype
# would make it incomparable to the rest. Prepare data on a T4 (RT_Benchmark_Prep.ipynb); score
# on L4 or better.
assert gpu["bf16_ok"], (
    "non-Ampere GPU (T4): no bf16 and no FlexAttention kernels, so scoring here is both very slow "
    "and not numerically comparable to the other arms. Use L4 or A100 -- with LEAN_MASKS an L4 is "
    "enough for the whole sweep including 30000.")
print("RLIMIT_NOFILE:", prep.raise_fd_limit())
prep.setup_scratch()

# Rewrite rt/model.py to build BlockMasks from mask_mods. Must happen BEFORE anything imports
# rt.model, and is proven equivalent to the stock dense construction before any scoring runs --
# a wrong mask would change which cells attend to which and quietly corrupt every metric.
if LEAN_MASKS:
    compat.patch_lean_block_masks(REPO)
    compat.selfcheck_block_masks()
else:
    print("[bench] LEAN_MASKS off: using the stock dense (B, S, S) masks")

# rustler's `pre` hardcodes RelBench v1 column fixups that panic on the v2 datasets this
# notebook installs. Patch the source BEFORE deciding whether the cached binary can be reused --
# a cache built from unpatched source is rejected by its PATCH_TAG and rebuilt.
prep.patch_rustler_relbench_v2(REPO)
RUSTLER = prep.restore_binary(PERSIST / "toolchain")
if RUSTLER is None:
    RUSTLER = prep.build_rustler(REPO)
    prep.save_toolchain(REPO, PERSIST / "toolchain")
if not prep.install_toolchain(PERSIST / "toolchain"):
    prep.build_rustler(REPO)
import rustler as _rustler_mod
print("rustler ready:", RUSTLER)

missing = [g for g, p in CHECKPOINTS.items() if not Path(p).expanduser().exists()]
assert not missing, (
    f"missing checkpoints: {missing}.\n"
    "If those arms are simply not trained YET, do not wait for them and do not retrain anything: "
    f"score the ones you have now by setting\n"
    f"    ONLY_GENERATORS = {[g for g in CHECKPOINTS if g not in missing]}\n"
    f"    SHARD           = '{'-'.join(g for g in CHECKPOINTS if g not in missing)}'\n"
    "and run the missing arm later as its own shard -- cell 6 merges every shard into one table."
)
print(f"{len(CHECKPOINTS)} checkpoints found: {sorted(CHECKPOINTS)}")


## 4. Materialize the RelBench v2 evaluation data

Downloads each database and its task tables into the layout `rustler pre` expects, then preprocesses and embeds. **Downloads are large** (rel-amazon is several GB); cached and resumable.

In [ ]:
#@title Evaluation data: restore from Drive, or build it and archive
tasks = bench.tasks_for(EVAL_DBS)
by_db = {}
for db, t, *_ in tasks:
    by_db.setdefault(db, []).append(t)

print(f"{len(tasks)} scorable tasks across {len(by_db)} databases")
for db, ts in by_db.items():
    print(f"  {db:14s} {len(ts)}  {ts}")
print(f"\nexcluded (no matching RT head):")
for k, v in bench.UNSCORABLE.items():
    print(f"  {k:46s} {v}")
print("")

# Preparation is the expensive half and `~/scratch` does not survive the session, so each
# database is archived to Drive the first time and restored (minutes, not hours) after that.
# Run RT_Benchmark_Prep.ipynb once on a cheap GPU and every session below hits the archive.
ARCHIVE = PERSIST / "benchmark" / "pre"
for db, ts in by_db.items():
    info = bench.prepare_or_restore(db, ts, RUSTLER, ARCHIVE)
    print(f"[bench] {db:14s} ready ({info['source']})")

# Classification labels must be BOOLEAN cells. rustler's cast-to-bool fixups in pre.rs are keyed
# on RelBench **v1** task-table names, so for a v2 task whose name differs the cast never runs and
# the label stays integer -- which makes it a numeric cell, leaving RT's boolean head nothing to
# score. Checking here costs a second; otherwise it surfaces only as status="single-class" after
# the GPU time is already spent.
import pandas as pd
notbool = []
for _db, _t, _col, _kind, _ in tasks:
    if _kind != "clf":
        continue
    _p = Path(os.environ["HOME"]) / "scratch" / "relbench" / _db / "tasks" / _t / f"{SPLIT}.parquet"
    if not _p.exists():
        continue
    _dt = str(pd.read_parquet(_p, columns=[_col])[_col].dtype)
    if _dt not in ("bool", "boolean"):
        notbool.append(f"{_db}/{_t}.{_col} is {_dt}")
print("")
if notbool:
    print(f"  !! {len(notbool)} classification label(s) are not boolean:")
    for _x in notbool:
        print(f"       {_x}")
    print("     rustler did not cast these, so they are NUMERIC cells and the boolean head has no")
    print("     label -- expect status='single-class', auroc=nan for those rows. The regression")
    print("     tasks and the other databases are unaffected, so the sweep is still worth running.")
else:
    print(f"all {sum(1 for t in tasks if t[3] == 'clf')} classification labels are boolean")


## 5. Run the sweep

Every (checkpoint x task x context length). Results are written to Drive after each row, so an interrupted session keeps what it measured.

In [ ]:
#@title Benchmark sweep
OUT = PERSIST / "benchmark" / f"{RESULT_FILE}.csv"
df = bench.run_benchmark(
    CHECKPOINTS,
    tasks=tasks,
    ctx_lens=tuple(CTX_LENS),
    out_csv=OUT,
    max_samples=MAX_SAMPLES,
    num_workers=NUM_WORKERS,
    split=SPLIT,
    seed=SEED,
    lean_masks=LEAN_MASKS,
)
print(f"\n{len(df)} rows -> {OUT}")
print(df["status"].value_counts().to_string())
print("")
print("predictions per cell (n) -- the basis for every metric below:")
ok_n = df[df["status"] == "ok"]
if len(ok_n):
    print(ok_n.groupby("ctx_len")["n"].describe()[["count", "min", "mean", "max"]].to_string())


## 6. Results

Mean AUROC (classification) and R² (regression) per generator x context length, plus the per-task detail.

In [ ]:
#@title Summary  (merges every shard of this sweep found on Drive)
import pandas as pd
pd.set_option("display.width", 220)

# Pulls in the other shards' CSVs too, so this reads as the whole sweep however it was split.
# Safe to run in every shard: `ok` beats a failed cell and duplicates collapse on
# (generator, db, task, ctx_len), so the last shard to finish prints the complete picture.
MERGED = PERSIST / "benchmark" / f"{RESULT_NAME}_merged.csv"
df = bench.merge_results(PERSIST / "benchmark" / f"{RESULT_NAME}*.csv", out_csv=MERGED)

summary = bench.summarize(df)
print(summary.round(4).to_string())
summary.to_csv(PERSIST / "benchmark" / f"{RESULT_NAME}_summary.csv")

print("\n\nper-task detail (ok rows only):")
ok = df[df["status"] == "ok"]
for kind, col in (("clf", "auroc"), ("reg", "r2")):
    sub = ok[ok["kind"] == kind]
    if sub.empty:
        continue
    piv = sub.pivot_table(index=["db", "task"], columns=["generator", "ctx_len"], values=col)
    print(f"\n=== {kind} ({col}) ===")
    print(piv.round(4).to_string())

bad = df[df["status"] != "ok"]
if len(bad):
    print(f"\n{len(bad)} non-ok cells:")
    print(bad.groupby(["status", "ctx_len"]).size().to_string())
